In [1]:
import jax
import jax.numpy as jnp

In [2]:
def assert_sym(M):
    assert jnp.all(M == M.T) or jnp.all(jnp.isclose(M, M.T, atol=1e-6)), "Not symmmetric!"

def symmetrize(M):
    return 0.5 * (M+M.T)

def assert_pd(M):
    M = symmetrize(M)
    assert jnp.all(jnp.linalg.eigvalsh(M) > 0), "Not PD!"
    
def assert_spd(M):
    assert_sym(M)
    assert_pd(M)

In [3]:
_key = lambda x: jax.random.PRNGKey(x)
def random_spd(key, d, /):
    Eps = jax.random.normal(key, (d,d))
    return Eps @ Eps.T# + jnp.eye(d)

D = 100

m = jax.random.normal(_key(112), (1,D))
C = random_spd(_key(911), D)  
A = random_spd(_key(114), D)  
# B = random_spd(_key(1813), D) 
B = jnp.eye(D) * .5

assert_spd(A)
assert_spd(B)
assert_spd(C)

C_sqrt = jnp.linalg.cholesky(C, upper=True)
B_sqrt = jnp.linalg.cholesky(B, upper=True)

sampling_dist = jax.random.rademacher
# sampling_dist = jax.random.normal

def sample_x(num_samples, /, key):
    N = sampling_dist(key=key, shape=(num_samples, D))
    return N@C_sqrt + m


def sample_y_given_x(x0, /, key):
    num_samples = x0.shape[0]
    N = sampling_dist(key=key, shape=(num_samples, D))
    return N@B_sqrt + x0@A


meow = C@A.T
meow2 = (A @ meow + B)
def K_apply_dense(v):
    return meow @ jax.scipy.linalg.solve(meow2.T, v.T)


y = jnp.zeros(1)                                                # TODO What is y? Synthetic data?
def sample_x_given_y(num_samples, /, key):
    key_x, key_y = jax.random.split(key)
    x0 = sample_x(num_samples, key=key_x)
    y0 = sample_y_given_x(x0, key=key_y)
    w  = K_apply_dense(y0 - y)
    return x0 - w


In [4]:
from matfree.lstsq import lsmr

# def meow_apply(v):
#     return meow2 @ v

B_sqrt_inv = 1 / B_sqrt # Since B is diagonal
meow3 = B_sqrt_inv @ A @ C_sqrt
def meow_apply(v):
    return meow3 @ v

solve = lsmr(atol=1e-3,btol=1e-3,ctol=1e-4)
def K_apply_mf(Vs):
    def body(v):
        y = C_sqrt @ v
        xi, info = solve(meow_apply, y)
        x = C_sqrt @ xi
        return x
    return jax.vmap(body)(Vs)
    # return jax.lax.map(body, Vs)

def sample_x_given_y_mf(num_samples, /, key):
    key_x, key_y = jax.random.split(key)
    x0 = sample_x(num_samples, key=key_x)
    y0 = sample_y_given_x(x0, key=key_y)
    w = K_apply_mf(y0 - y)
    return x0 - w

In [5]:
a_lot_of_samples = sample_x_given_y_mf(1_000, _key(117))

In [6]:
# analytical_mean = m - K_apply_dense((m@A - y).squeeze())
analytical_mean = m - K_apply_mf((m@A - y))
print("======MEAN======")
print(a_lot_of_samples.mean(0))
print(analytical_mean.squeeze())

print("======VAR=======")
# analytical_covariance = C - K_apply_dense(meow.squeeze())
analytical_covariance = C - K_apply_mf(meow)
print(a_lot_of_samples.var(0))
print(jnp.diag(analytical_covariance))

======MEAN======
[-1.2172991  -1.7142859   1.0854659  -0.871795    0.1821639   0.12748586
 -1.7147793   1.6813307  -0.25934806 -0.26360726 -2.1909065  -1.2742512
  0.18631579  0.92858535 -2.3719501  -0.28474078  0.9097582   0.7798806
 -1.7278261  -0.1932787   0.7841905  -0.26734412  0.40062132  0.01382642
  0.38975716  0.36700043  0.40911156  0.20307408 -0.56783515 -2.2963238
 -1.0332177  -1.0446255  -0.07067811 -0.30713627 -0.6449304  -0.3737237
  0.7807069   0.24330302 -0.4004224  -2.7481136  -0.3714111  -0.719212
  1.4894422   0.39345285  1.5265363   1.7081008   0.5215977  -1.0076135
 -0.8604385  -0.76626927 -0.57093495  0.01633944 -0.94122934 -1.1147501
  0.5755794   0.37719083  0.37775135  0.7513671   0.25923082  1.3153905
 -0.69696665  0.31604868 -0.07932475 -0.06082314 -0.32666475 -0.7971148
  0.37031108 -1.5872242   1.1903908  -0.35035667  1.6674259  -1.2026621
 -2.72537     0.16792911 -0.97128296  0.480602   -0.6631415  -0.44168016
  0.01908337  1.6594875   0.2640602   0.08951

In [7]:
t1 = (a_lot_of_samples - a_lot_of_samples.mean(0, keepdims=True))
sample_cov = t1.T@t1 / (a_lot_of_samples.shape[0] - 1)

jnp.linalg.trace(sample_cov), jnp.linalg.trace(analytical_covariance)

(Array(9947.69, dtype=float32), Array(9922.811, dtype=float32))

In [8]:
S = random_spd(_key(72), D)
Si = jnp.linalg.inv(S)

jnp.linalg.trace(Si@analytical_covariance)

Array(15094.11, dtype=float32)

In [9]:
from jax.flatten_util import ravel_pytree
from matfree import stochtrace

problem = stochtrace.integrand_trace()

def sampler_lla(*args_like, num):
    # TODO pass a bunch of stuff that makes sense in the LLA / GGN context
    x_flat, unflatten = ravel_pytree(*args_like)
    def sampler(key):
        return sample_x_given_y_mf(num, key)
    return sampler


sampler = sampler_lla(m, num=100_000)
estimate = stochtrace.estimator(problem, sampler)
estimate = jax.jit(estimate, static_argnums=[0])
estimate(lambda v: Si@v, key=_key(123444))

Array(15175.047, dtype=float32)

In [10]:
estimate(lambda v: Si@v, key=_key(4321))

Array(15172.137, dtype=float32)

In [12]:
sampler = stochtrace.sampler_normal(jnp.ones(D), num=100_000)
estimate = stochtrace.estimator(problem, sampler)
estimate = jax.jit(estimate, static_argnums=[0])
def Si_apply(v):
    x,info = jax.scipy.sparse.linalg.cg(S, v)
    return x
estimate(lambda v: analytical_covariance@Si_apply(v), key=_key(123422444))
# estimate(lambda v: Si@v, key=_key(123))

Array(15050.575, dtype=float32)

In [13]:
estimate(lambda v: analytical_covariance@Si_apply(v), key=_key(70_421))

Array(15097.067, dtype=float32)